# Drone RL Navigation — Training Analysis

This notebook provides end-to-end analysis of the PPO training run:
- Reward curves & convergence
- Reward component breakdown
- Goal-success and collision-rate trends
- 3-D flight trajectory visualisation
- Evaluation statistics

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # allow imports from project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from utils.visualization import (
    plot_training_rewards,
    plot_reward_components,
    plot_episode_trajectory,
    plot_evaluation_summary,
    plot_sensor_heatmap,
)
from config.env_config import ENV_CONFIG
from config.training_config import TRAIN_CONFIG

print('Imports OK')

## 1. Load Training Logs

In [ ]:
LOG_CSV = "../logs/run_latest.csv"   # ← adjust to your run file

try:
    df = pd.read_csv(LOG_CSV)
    print(f"Loaded {len(df)} log rows.")
    display(df.head())
except FileNotFoundError:
    print(f"Log file not found at {LOG_CSV}. Run training first or adjust the path.")
    df = pd.DataFrame()  # empty placeholder

## 2. Reward Curve

In [ ]:
if 'episode/total_reward' in df.columns:
    rewards = df['episode/total_reward'].dropna().tolist()
    fig = plot_training_rewards(rewards, window=50, title="Training Reward Curve")
    plt.show()
else:
    # Demo with synthetic data
    np.random.seed(42)
    synthetic = np.cumsum(np.random.randn(500)) - np.linspace(50, -100, 500)
    fig = plot_training_rewards(synthetic.tolist(), window=50, title="Synthetic Demo Curve")
    plt.show()

## 3. Goal-Success & Collision Rate

In [ ]:
if 'episode/goal_reached' in df.columns:
    window = 50
    goal_rate  = df['episode/goal_reached'].rolling(window).mean()
    crash_rate = df['episode/collision'].rolling(window).mean()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(goal_rate.values,  label='Goal rate',      color='#81c784', lw=2)
    ax.plot(crash_rate.values, label='Collision rate', color='#f06292', lw=2)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Rate (rolling 50-ep)')
    ax.set_title('Goal Success vs Collision Rate')
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.show()
else:
    print("No episode/goal_reached column found — run training to generate logs.")

## 4. Reward Component Breakdown

In [ ]:
# Synthetic reward components for demonstration
np.random.seed(0)
n_steps = 300
components = [
    {
        'reward/progress':  max(0, np.random.normal(1.5, 0.5)),
        'reward/step':      0.1,
        'reward/altitude':  max(0, np.random.normal(0.2, 0.1)),
    }
    for _ in range(n_steps)
]

fig = plot_reward_components(components)
plt.show()

## 5. 3-D Flight Trajectory

In [ ]:
TRAJ_CSV = "../eval_results/trajectories.csv"

try:
    traj_df = pd.read_csv(TRAJ_CSV)
    ep0     = traj_df[traj_df['episode'] == 0]
    positions = list(zip(ep0['x'], ep0['y'], ep0['z']))
    print(f"Loaded trajectory with {len(positions)} waypoints.")
except FileNotFoundError:
    # Synthetic helical path for demonstration
    t = np.linspace(0, 2*np.pi, 200)
    positions = [
        (5*np.cos(s) + s*30/(2*np.pi),
         5*np.sin(s),
         -5 - 3*s/(2*np.pi))
        for s in t
    ]
    print("Using synthetic helical trajectory for demo.")

fig = plot_episode_trajectory(
    positions,
    target=ENV_CONFIG.target_position,
    spawn=ENV_CONFIG.spawn_position,
)
plt.show()

## 6. Distance-Sensor Heat-map

In [ ]:
np.random.seed(7)
steps, n_sensors = 300, 8
# Simulate sensors decreasing as drone approaches obstacles, then recovering
base = np.random.uniform(8, 20, (steps, n_sensors))
base[80:120, [0, 1]] *= 0.3   # front sensors see an obstacle
base[180:200, [3, 4]] *= 0.4
sensor_history = np.clip(base, 0, 20)

fig = plot_sensor_heatmap(sensor_history, sensor_max=20.0)
plt.show()

## 7. Evaluation Statistics

In [ ]:
import json

EVAL_JSON = "../eval_results/eval_stats.json"

try:
    with open(EVAL_JSON) as f:
        stats = json.load(f)
except FileNotFoundError:
    # Placeholder stats for demonstration
    stats = {
        'success_rate':     0.72,
        'collision_rate':   0.15,
        'mean_reward':      87.4,
        'std_reward':       22.1,
        'mean_steps':       183.0,
        'mean_path_length': 34.8,
    }
    print("Loaded placeholder statistics.")

fig = plot_evaluation_summary(stats)
plt.show()

print("\n=== Summary ===")
for k, v in stats.items():
    print(f"  {k:<25} {v}")

## 8. Quick Sanity Check — Environment & Policy


In [ ]:
from environment.airsim_env import DroneNavigationEnv
from environment.state_processing import StateProcessor
from environment.reward_function import RewardFunction
from agent.model import DroneActorCritic
import torch

# Instantiate environment (uses mock client when AirSim is not running)
env = DroneNavigationEnv()
obs, _ = env.reset()
print(f"Observation shape : {obs.shape}")
print(f"Action space      : {env.action_space}")

# Step with a random action
action = env.action_space.sample()
obs2, reward, terminated, truncated, info = env.step(action)
print(f"Step reward       : {reward:.4f}")
print(f"Info keys         : {list(info.keys())}")
env.close()

# Instantiate standalone model
sp = StateProcessor()
model = DroneActorCritic(obs_dim=sp.obs_dim)
print(f"\nPolicy parameters : {model.count_parameters():,}")

dummy_obs = torch.zeros(1, sp.obs_dim)
mean, log_std, value = model(dummy_obs)
print(f"Action mean       : {mean.detach().numpy()}")
print(f"Value estimate    : {value.item():.4f}")